In [ ]:
import json
import os
import time
from pathlib import Path

import pandas as pd
import requests
from tqdm.auto import tqdm
from getpass import getpass


# Paths

project_directory = Path(
    r"C:\Users\nikola.bakic\OneDrive - Sixsentix AG\Documents"
    r"\IMDB_WEEK1_SIX\movie-embeddings-project"
)

dataset_path = (
    project_directory
    / "data"
    / "processed"
    / "movies_cleaned.csv"
)

metadata_directory = (
    project_directory
    / "artifacts"
    / "metadata"
)

metadata_directory.mkdir(
    parents=True,
    exist_ok=True,
)

genres_path = metadata_directory / "movie_genres.csv"
progress_path = metadata_directory / "movie_genres_progress.csv"


# TMDB authentication

tmdb_token = os.environ.get("TMDB_BEARER_TOKEN")

if not tmdb_token:
    tmdb_token = getpass(
        "TMDB_BEARER_TOKEN nije postavljen." \
        "Unesite TMDB_BEARER_TOKEN: "
    )

headers = {
    "Authorization": f"Bearer {tmdb_token}",
    "accept": "application/json",
}


# Load movies

movies_df = pd.read_csv(dataset_path)

required_columns = {"movie_id", "title"}
missing_columns = required_columns - set(movies_df.columns)

assert not missing_columns, (
    f"Nedostaju kolone: {sorted(missing_columns)}"
)

assert movies_df["movie_id"].notna().all()
assert movies_df["movie_id"].is_unique

movies_df["movie_id"] = movies_df["movie_id"].astype(int)


# Resume previous progress if it exists

if progress_path.exists():
    existing_df = pd.read_csv(progress_path)

    completed_movie_ids = set(
        existing_df["movie_id"].astype(int)
    )

    results = existing_df.to_dict("records")

    print(
        f"Nastavlja se od prethodnog progresa: "
        f"{len(completed_movie_ids)} filmova."
    )
else:
    completed_movie_ids = set()
    results = []


# HTTP session

session = requests.Session()
session.headers.update(headers)


def fetch_movie_genres(
    movie_id: int,
    max_retries: int = 5,
) -> dict:
    url = f"https://api.themoviedb.org/3/movie/{movie_id}"

    for attempt in range(max_retries):
        try:
            response = session.get(
                url,
                timeout=30,
            )

            if response.status_code == 200:
                data = response.json()

                genres = data.get("genres", [])

                return {
                    "movie_id": movie_id,
                    "tmdb_title": data.get("title"),
                    "genre_ids": json.dumps(
                        [genre["id"] for genre in genres]
                    ),
                    "genres": json.dumps(
                        [genre["name"] for genre in genres],
                        ensure_ascii=False,
                    ),
                    "genre_count": len(genres),
                    "status": "success",
                }

            if response.status_code == 404:
                return {
                    "movie_id": movie_id,
                    "tmdb_title": None,
                    "genre_ids": "[]",
                    "genres": "[]",
                    "genre_count": 0,
                    "status": "not_found",
                }

            if response.status_code == 429:
                retry_after = int(
                    response.headers.get("Retry-After", 2)
                )
                time.sleep(retry_after)
                continue

            response.raise_for_status()

        except requests.RequestException as error:
            if attempt == max_retries - 1:
                return {
                    "movie_id": movie_id,
                    "tmdb_title": None,
                    "genre_ids": "[]",
                    "genres": "[]",
                    "genre_count": 0,
                    "status": f"error: {type(error).__name__}",
                }

            time.sleep(2 ** attempt)

    return {
        "movie_id": movie_id,
        "tmdb_title": None,
        "genre_ids": "[]",
        "genres": "[]",
        "genre_count": 0,
        "status": "failed",
    }


# Download genre metadata

movies_to_process = movies_df[
    ~movies_df["movie_id"].isin(completed_movie_ids)
][["movie_id", "title"]]

for index, row in tqdm(
    movies_to_process.iterrows(),
    total=len(movies_to_process),
    desc="Preuzimanje TMDB žanrova",
):
    result = fetch_movie_genres(
        movie_id=int(row["movie_id"]),
    )

    result["dataset_title"] = row["title"]
    results.append(result)

    # Save every 100 requests so the process can resume.
    if len(results) % 100 == 0:
        pd.DataFrame(results).to_csv(
            progress_path,
            index=False,
        )


# Save final result

genres_df = pd.DataFrame(results)

genres_df = (
    genres_df
    .drop_duplicates(
        subset="movie_id",
        keep="last",
    )
    .sort_values("movie_id")
    .reset_index(drop=True)
)

genres_df.to_csv(
    genres_path,
    index=False,
)

# Keep the same file as resumable progress.
genres_df.to_csv(
    progress_path,
    index=False,
)

print(f"Ukupno filmova: {len(genres_df)}")
print("\nStatus:")
print(genres_df["status"].value_counts(dropna=False))

print(
    f"\nFilmovi sa najmanje jednim žanrom: "
    f"{(genres_df['genre_count'] > 0).sum()}"
)

print(f"\nSačuvano u:\n{genres_path}")

display(genres_df.head())

c:\Users\nikola.bakic\OneDrive - Sixsentix AG\Documents\IMDB_WEEK1_SIX\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Preuzimanje TMDB žanrova: 100%|██████████| 9967/9967 [25:36<00:00,  6.49it/s]   


Ukupno filmova: 9967

Status:
status
success    9967
Name: count, dtype: int64

Filmovi sa najmanje jednim žanrom: 9965

Sačuvano u:
C:\Users\nikola.bakic\OneDrive - Sixsentix AG\Documents\IMDB_WEEK1_SIX\movie-embeddings-project\artifacts\metadata\movie_genres.csv


,movie_id,tmdb_title,genre_ids,genres,genre_count,status,dataset_title
0,2,Ariel,"[35, 18, 10749, 80]","[""Comedy"", ""Drama"", ""Romance"", ""Crime""]",4,success,Ariel
1,3,Shadows in Paradise,"[35, 18, 10749]","[""Comedy"", ""Drama"", ""Romance""]",3,success,Shadows in Paradise
2,5,Four Rooms,[35],"[""Comedy""]",1,success,Four Rooms
3,6,Judgment Night,"[28, 80, 53]","[""Action"", ""Crime"", ""Thriller""]",3,success,Judgment Night
4,11,Star Wars,"[12, 28, 878]","[""Adventure"", ""Action"", ""Science Fiction""]",3,success,Star Wars


In [3]:
genre_check_df = movies_df[
    ["movie_id", "title"]
].merge(
    genres_df[
        [
            "movie_id",
            "tmdb_title",
            "genres",
            "genre_count",
            "status",
        ]
    ],
    on="movie_id",
    how="left",
    validate="one_to_one",
)

genre_check_df["title_match"] = (
    genre_check_df["title"]
    .astype(str)
    .str.strip()
    .str.casefold()
    ==
    genre_check_df["tmdb_title"]
    .astype(str)
    .str.strip()
    .str.casefold()
)

print(
    genre_check_df[
        "status"
    ].value_counts(dropna=False)
)

print(
    f"Tačno poklapanje naslova: "
    f"{genre_check_df['title_match'].mean():.2%}"
)

display(
    genre_check_df[
        ~genre_check_df["title_match"]
    ].head(20)
)

status
success    9967
Name: count, dtype: int64
Tačno poklapanje naslova: 99.86%


,movie_id,title,tmdb_title,genres,genre_count,status,title_match
350,81,Nausicaä of the Valley of the Wind,Warriors of the Wind,"[""Adventure"", ""Animation"", ""Fantasy""]",3,success,False
1449,676701,Tad and The Emerald Tablet,Tad the Lost Explorer and the Emerald Tablet,"[""Animation"", ""Adventure"", ""Family"", ""Comedy"",...",5,success,False
1801,615173,The Witch Part 2 - The Other One,The Witch: Part 2. The Other One,"[""Action"", ""Mystery"", ""Thriller"", ""Science Fic...",4,success,False
1979,619803,The Roundup 2,The Roundup,"[""Action"", ""Crime"", ""Thriller"", ""Adventure""]",4,success,False
2009,25403,Dear Diary,Caro diario,"[""Comedy"", ""Drama""]",2,success,False
2508,12598,Joe: The Busybody,Jo,"[""Comedy"", ""Crime""]",2,success,False
2588,955555,The Roundup 3: No Way Out,The Roundup: No Way Out,"[""Action"", ""Crime"", ""Comedy"", ""Thriller""]",4,success,False
2841,505954,T-34,Т-34,"[""Drama"", ""History"", ""War""]",3,success,False
3132,1584,School of Rock,The School of Rock,"[""Comedy"", ""Family"", ""Music""]",3,success,False
3898,760336,Munich – The Edge of War,Munich - The Edge of War,"[""Drama"", ""History""]",2,success,False
